In [1]:

import numpy as np
import sys
%load_ext autoreload
%autoreload 2
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent)) 
from commom_utils.systems import *
from commom_utils.ode_system import ODESystem, check_system_ok, SyntheticDataGenerator
from commom_utils.system_config import create_system, SYSTEM_CONFIGS
from gauss_newton.utils import plot_solution
import matplotlib.pyplot as plt
from gauss_newton.problem import MultipleShooting
from gauss_newton.adaptive import run_optimization_adaptive
from scipy.interpolate import interp1d
from experiments.data_utils import  LogReaderV2, create_interval_batches

In [2]:
# main_dir = Path("/home/iachichkanov/autotech/GaussNewton/experiments/voyax_free/Lateral Dynamics/50 kph")
root_dir = Path("/home/iachichkanov/autotech/GaussNewton/experiments/voyax_free/GearRatioCheck")
# main_dir = root_dir/"Right"
# data_storage_EPS_angle = LogReader(main_dir/"EPS_angle.csv")
# data_storage_ESP_IMU = LogReader(main_dir/"ESP_IMU.csv")
# data_storage_IMU_2 = LogReader(main_dir/"IMU_2.csv")
# data_storage_wheel_speed = LogReader(main_dir/"ESP_1_rear_speed_wheel.csv")

In [29]:
import os
import pandas as pd
from pathlib import Path
import glob
from collections import defaultdict
def merge_csv_sequential(source_folders, output_folder, time_column='time', 
                         method='linear', sample_rate=None):
    """
    Последовательно объединяет CSV файлы, убирая зазоры между ними
    Время становится непрерывным от 0 до суммарной длительности
    """
    os.makedirs(output_folder, exist_ok=True)
    
    file_groups = defaultdict(list)
    for folder in source_folders:
        folder_path = Path(folder)
        if not folder_path.exists():
            continue
        for file in folder_path.glob("*.csv"):
            file_groups[file.name].append(file)
    
    for filename, file_paths in file_groups.items():
        if len(file_paths) <= 1:
            continue
            
        print(f"\nSequential merge {filename} from {len(file_paths)} sources...")
        
        # Читаем и сортируем файлы по времени начала
        file_info = []
        for file_path in file_paths:
            df = pd.read_csv(file_path)
            file_info.append({
                'path': file_path,
                'df': df,
                'source': file_path.parent.name,
                't_start': df[time_column].min(),
                't_end': df[time_column].max(),
                'duration': df[time_column].max() - df[time_column].min()
            })
        
        # Сортируем по времени начала
        file_info.sort(key=lambda x: x['t_start'])
        
        print("  Files order:")
        for info in file_info:
            print(f"    {info['source']}: [{info['t_start']:.3f}, {info['t_end']:.3f}], "
                  f"duration={info['duration']:.3f}")
        
        # Находим общие колонки
        common_columns = set(file_info[0]['df'].columns)
        for info in file_info[1:]:
            common_columns &= set(info['df'].columns)
        common_columns.discard(time_column)
        
        # Создаем новое непрерывное время
        time_offset = 0
        all_segments = []
        
        for i, info in enumerate(file_info):
            df = info['df'].copy()
            
            # Нормализуем время относительно начала этого файла
            df[time_column] = df[time_column] - info['t_start']
            
            # Добавляем смещение для непрерывности
            df[time_column] = df[time_column] + time_offset
            
            # Добавляем колонку с источником (опционально)
            df['source'] = info['source']
            
            all_segments.append(df)
            
            # Обновляем смещение для следующего файла
            time_offset += info['duration']
            
            # Убираем зазор: начинаем следующий файл сразу после окончания текущего
            if i < len(file_info) - 1:
                gap = file_info[i+1]['t_start'] - info['t_end']
                if gap > 0:
                    print(f"    Gap of {gap:.3f}s removed between {info['source']} and {file_info[i+1]['source']}")
        
        # Объединяем все сегменты
        merged_df = pd.concat(all_segments, ignore_index=True)
        
        # Опционально: ресемплинг на равномерную сетку
        if sample_rate:
            t_min = merged_df[time_column].min()
            t_max = merged_df[time_column].max()
            common_time = np.arange(t_min, t_max, sample_rate)
            
            resampled_df = pd.DataFrame({time_column: common_time})
            
            for col in common_columns:
                # Сортируем и убираем дубликаты
                df_sorted = merged_df.sort_values(time_column).drop_duplicates(subset=[time_column])
                
                # Интерполируем
                interp_func = interp1d(
                    df_sorted[time_column], df_sorted[col],
                    kind=method, bounds_error=False, fill_value=np.nan
                )
                resampled_df[col] = interp_func(common_time)
            
            # Добавляем source (берем из ближайшего времени)
            source_interp = interp1d(
                df_sorted[time_column], 
                pd.Categorical(df_sorted['source']).codes,
                kind='nearest', bounds_error=False
            )
            categories = df_sorted['source'].unique()
            resampled_df['source'] = categories[source_interp(common_time).astype(int)]
            
            merged_df = resampled_df.dropna()
        
        # Сохраняем
        output_path = os.path.join(output_folder, filename)
        merged_df.to_csv(output_path, index=False)
        print(f"  Saved: {output_path}")
        print(f"  Total duration: {time_offset:.3f}s")
        print(f"  Total points: {len(merged_df)}")


# Использование:
merge_csv_sequential(
    [root_dir/"Right", root_dir/"Left"], 
    root_dir/"all",
    sample_rate=0.01  # 100 Гц
)




Sequential merge ESP_IMU.csv from 2 sources...
  Files order:
    Left: [1780914734.247, 1780915102.022], duration=367.776
    Right: [1780915203.421, 1780915745.614], duration=542.193
    Gap of 101.399s removed between Left and Right
  Saved: /home/iachichkanov/autotech/GaussNewton/experiments/voyax_free/GearRatioCheck/all/ESP_IMU.csv
  Total duration: 909.969s
  Total points: 90997

Sequential merge EPS_angle.csv from 2 sources...
  Files order:
    Left: [1780914734.238, 1780915102.040], duration=367.802
    Right: [1780915203.410, 1780915745.623], duration=542.213
    Gap of 101.370s removed between Left and Right
  Saved: /home/iachichkanov/autotech/GaussNewton/experiments/voyax_free/GearRatioCheck/all/EPS_angle.csv
  Total duration: 910.015s
  Total points: 91002

Sequential merge ESP_1_rear_speed_wheel.csv from 2 sources...
  Files order:
    Left: [1780914734.245, 1780915102.040], duration=367.796
    Right: [1780915203.419, 1780915745.612], duration=542.193
    Gap of 101.37

In [3]:
main_dir = root_dir/"all"
data_storage_EPS_angle = LogReaderV2(main_dir/"EPS_angle.csv")
data_storage_ESP_IMU = LogReaderV2(main_dir/"ESP_IMU.csv")
data_storage_IMU_2 = LogReaderV2(main_dir/"IMU_2.csv")
data_storage_wheel_speed = LogReaderV2(main_dir/"ESP_1_rear_speed_wheel.csv")

Loaded 91002 rows from /home/iachichkanov/autotech/GaussNewton/experiments/voyax_free/GearRatioCheck/all/EPS_angle.csv
Columns: ['time', 'EPS_SAS_SteerAngleSpdValid', 'h144_checksum', 'EPS_SAS_SteerAngleValid', 'h144_undefine', 'h144_counter', 'source']
Loaded 90997 rows from /home/iachichkanov/autotech/GaussNewton/experiments/voyax_free/GearRatioCheck/all/ESP_IMU.csv
Columns: ['time', 'LongAx', 'YAW_Rate', 'LattAx', 'source']
Loaded 90997 rows from /home/iachichkanov/autotech/GaussNewton/experiments/voyax_free/GearRatioCheck/all/IMU_2.csv
Columns: ['time', 'LongAx', 'YAW_Rate', 'LattAx', 'source']
Loaded 90999 rows from /home/iachichkanov/autotech/GaussNewton/experiments/voyax_free/GearRatioCheck/all/ESP_1_rear_speed_wheel.csv
Columns: ['time', 'IPB_wheelSpeedRL', 'h122_counter', 'h122_checksum', 'IPB_wheelSpeedRR', 'source']


In [4]:
print(data_storage_ESP_IMU.df_data.columns)
print(data_storage_EPS_angle.df_data.columns)
print(data_storage_wheel_speed.df_data.columns)
print(data_storage_IMU_2.df_data.columns)

Index(['time', 'LongAx', 'YAW_Rate', 'LattAx', 'source'], dtype='object')
Index(['time', 'EPS_SAS_SteerAngleSpdValid', 'h144_checksum',
       'EPS_SAS_SteerAngleValid', 'h144_undefine', 'h144_counter', 'source'],
      dtype='object')
Index(['time', 'IPB_wheelSpeedRL', 'h122_counter', 'h122_checksum',
       'IPB_wheelSpeedRR', 'source'],
      dtype='object')
Index(['time', 'LongAx', 'YAW_Rate', 'LattAx', 'source'], dtype='object')


In [18]:
data_storage_EPS_angle.add_batch( "EPS_SAS_SteerAngleValid", use_jax_interp=0, scale_coef = np.deg2rad(1))
data_storage_ESP_IMU.add_batch( "LattAx", use_jax_interp=0, scale_coef = -1)
data_storage_ESP_IMU.add_batch( "YAW_Rate", use_jax_interp=0,  scale_coef = -np.deg2rad(1))
data_storage_IMU_2.add_batch( "YAW_Rate", use_jax_interp=0,  scale_coef = -np.deg2rad(1))
data_storage_wheel_speed.add_batch("IPB_wheelSpeedRL", use_jax_interp=0, scale_coef = 1.0/3.6)
data_storage_wheel_speed.add_batch("IPB_wheelSpeedRR", use_jax_interp=0, scale_coef = 1.0/3.6)

data_storage_EPS_angle.process_all()
data_storage_ESP_IMU.process_all()
data_storage_IMU_2.process_all()
data_storage_wheel_speed.process_all()
#t = data_storage_EPS_angle.get_time("EPS_SAS_SteerAngleValid")
f_steer = data_storage_EPS_angle.get_f_interp("EPS_SAS_SteerAngleValid")
f_lat_ax = data_storage_ESP_IMU.get_f_interp("LattAx")
f_yaw_rate1 = data_storage_ESP_IMU.get_f_interp("YAW_Rate")
f_yaw_rate2 = data_storage_IMU_2.get_f_interp("YAW_Rate")
f_wheelSpeedRL = data_storage_wheel_speed.get_f_interp("IPB_wheelSpeedRL")
f_wheelSpeedRR = data_storage_wheel_speed.get_f_interp("IPB_wheelSpeedRR")


Queued EPS_SAS_SteerAngleValid: time [0.000, 910.010], values [-8.772, 8.831]
Queued LattAx: time [0.000, 909.960], values [-2.030, 1.887]
Queued YAW_Rate: time [0.000, 909.960], values [-0.621, 0.622]
Queued YAW_Rate: time [0.000, 909.960], values [-0.616, 0.625]
Queued IPB_wheelSpeedRL: time [0.000, 909.980], values [0.000, 7.980]
Queued IPB_wheelSpeedRR: time [0.000, 909.980], values [0.000, 8.272]

Set t0 = 0.0 (minimum of all start times)
Processed EPS_SAS_SteerAngleValid: original [0.000, 910.010], normalized [0.000, 910.010]
Common time range: [0.000, 910.010]

Set t0 = 0.0 (minimum of all start times)
Processed LattAx: original [0.000, 909.960], normalized [0.000, 909.960]
Processed YAW_Rate: original [0.000, 909.960], normalized [0.000, 909.960]
Common time range: [0.000, 909.960]

Set t0 = 0.0 (minimum of all start times)
Processed YAW_Rate: original [0.000, 909.960], normalized [0.000, 909.960]
Common time range: [0.000, 909.960]

Set t0 = 0.0 (minimum of all start times)
Pr

In [19]:
[data_storage_EPS_angle.t2, data_storage_ESP_IMU.t2, data_storage_IMU_2.t2]

[np.float64(910.01), np.float64(909.96), np.float64(909.96)]

In [20]:
t2 = np.min([data_storage_EPS_angle.t2, data_storage_ESP_IMU.t2, data_storage_IMU_2.t2])
t1 = np.max([data_storage_EPS_angle.t1, data_storage_ESP_IMU.t1, data_storage_IMU_2.t1])
t = np.linspace(t1, t2 - 1, 4000)

In [11]:
f_vx =  interp1d(t, (f_wheelSpeedRL(t) + f_wheelSpeedRR(t))/2, kind = 'linear') 

In [12]:
from casadi import SX, vertcat, Function, jacobian, vertcat


    
import casadi as ca
import numpy as np
from abc import abstractmethod
from casadi import SX, vertcat, Function, jacobian

class Regressor:
    def __init__(self, system: ODESystem):
        self.system = system
        state_var, theta_var, inp_signal_var = system.state, system.theta, system.u
        h_observ = system.observation(state_var, theta_var, inp_signal_var)

        self.res_h = Function('h_x', [*state_var.elements(), *theta_var.elements(), *inp_signal_var.elements()], [h_observ])
        
        J_h_theta = jacobian(h_observ, theta_var)
        self.compute_jacobian_h_theta = Function('J_h_x', [*state_var.elements(), *theta_var.elements(), *inp_signal_var.elements()], [J_h_theta])

    def get_inp_signals_(self, t):
        try:
            inp_signals = self.system.get_input_signals(t)
        except ValueError:
            inp_signals = np.zeros(self.system.nu)
            print(f'df_dx interpolation error, time {t}')
        return inp_signals
    
    def h_x(self, t, state_measured, theta):
        t = np.atleast_1d(t)
        state_measured = np.atleast_2d(state_measured)
        
        results = []
        for i in range(len(t)):
            inp_signals = self.get_inp_signals_(t[i])
            res = np.array(self.res_h(*[*state_measured[i], *theta, *inp_signals])).T[0]
            results.append(res)
        
        result = np.array(results)
        return result.squeeze()
    
    def dh_dtheta_(self, t, state_measured, theta):
        t = np.atleast_1d(t)
        state_measured = np.atleast_2d(state_measured)
        
        results = []
        for i in range(len(t)):
            inp_signals = self.get_inp_signals_(t[i])
            res = np.array(self.compute_jacobian_h_theta(*[*state_measured[i], *theta, *inp_signals]))
            results.append(res)
        
        return np.array(results).squeeze()
    
    def solve(self, t, observation, theta):
        N_measurement = len(t)
        assert N_measurement == len(observation)
        
        R = np.zeros((N_measurement, self.system.n_obs))
        J = np.zeros((len(t), self.system.n_obs, self.system.n_theta))
        
        for ind, t_i in enumerate(t):
            state_measured = observation[ind]
            J[ind] = self.dh_dtheta_(t_i, state_measured, theta) 
            R[ind] = state_measured - self.h_x(t_i, state_measured, theta)

        J = J.reshape(N_measurement * self.system.n_obs, -1)
        R = R.reshape(N_measurement * self.system.n_obs)
        return J, R

In [17]:
wheelbase = 2.96
gear_ratio_test  = 0.07
rwa = f_steer(t)*gear_ratio_test
vx = f_vx(t)
plt.plot(t,  vx*np.tan(rwa)/(wheelbase))
plt.plot(t, f_yaw_rate1(t))
# plt.plot(t, vx)
plt.plot(t, vx*vx*rwa/wheelbase)
plt.plot(t, -f_lat_ax(t))
# plt.plot(t, f_steer(t))
state_measured = f_yaw_rate1(t)[:, np.newaxis]

# plt.plot(t, f_lat_ax(t))
# plt.plot(t, f_wheelSpeedRL(t))
# plt.plot(t, f_wheelSpeedRR(t))
# plt.plot(t, f_vx(t))
# plt.plot(t, f_yaw_rate1(t))
# plt.plot(t, f_yaw_rate2(t))

<Figure size 640x480 with 1 Axes>

In [39]:
rwa.min()

np.float64(-0.614041737436645)

In [40]:
class KinematicBycicle(ODESystem):
    def __init__(self, wheelbase):
        self.wheelbase = wheelbase
        super().__init__(nx=1, nu=2, n_theta=2)
        

    def get_derivative(self, state, params, input_signals):
        pass

    def observation(self, state, params, input_signals):
        GR = params[0]
        offset = params[1]
        v = input_signals[0]
        steering = input_signals[1]
        rwa = GR * steering + offset
        w = v * np.tan(rwa) / self.wheelbase
        return vertcat(w)
    
    def get_input_signals(self, t):
        return [f_vx(t), f_steer(t)]
    
system = KinematicBycicle(wheelbase)
theta = np.zeros(2)
# theta[0] = 1/20


# regressor = Regressor(system)

In [41]:
state_measured.shape

(4000, 1)

In [42]:
regressor = Regressor(system)
for i in range(5):
    J, R = regressor.solve(t, state_measured, theta)
    H = J.T@J
    print(H, R)
    print(theta)
    delta_theta_estimate = np.linalg.pinv(H + 0.01*np.diag(np.diag(H)))@J.T@R
    theta = theta + delta_theta_estimate


[[110043.0674395   -1264.69483975]
 [ -1264.69483975   8302.24402073]] [ 0.55064265  0.53730288  0.53451581 ... -0.00707731 -0.00602139
 -0.00449365]
[0. 0.]
[[184836.6213603   -2015.51362208]
 [ -2015.51362208  10036.10739709]] [-0.0063921  -0.0157101  -0.01849717 ... -0.00598496 -0.00494919
 -0.00343758]
[ 0.07608413 -0.00047168]
[[173483.74944278  -1912.29527787]
 [ -1912.29527787   9789.34438335]] [ 0.03490948  0.02529839  0.02251133 ... -0.00630443 -0.00526277
 -0.00374644]
[ 7.18302420e-02 -8.24316518e-05]
[[173142.81194097  -1911.28909931]
 [ -1911.28909931   9781.87301066]] [ 0.03625981  0.02663911  0.02385204 ... -0.00628569 -0.00524437
 -0.00372833]
[ 0.07169174 -0.00011068]
[[173137.28192438  -1911.3180083 ]
 [ -1911.3180083    9781.75194366]] [ 0.03628271  0.02666185  0.02387479 ... -0.00628479 -0.00524349
 -0.00372745]
[ 0.07168949 -0.00011198]


In [ ]:
fig = plt.figure(figsize=(13, 8))

# Основной график
ax1 = plt.subplot(2, 1, 1)  # 2 строки, 1 столбец, 1-й график
ax1.plot(t, regressor.h_x(t, state_measured, theta), label="regression", alpha=0.8)
ax1.plot(t, state_measured, label="measured", alpha=0.8)
ax1.set_xlabel('Time')
ax1.set_ylabel('Yaw rate')
ax1.set_title('Very Simple Kinematic model')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Гистограмма ошибок
ax2 = plt.subplot(2, 1, 2)  # 2 строки, 1 столбец, 2-й график

# Вычисляем ошибки
errors = state_measured.flatten() - regressor.h_x(t, state_measured, theta).flatten()

ax2.hist(errors, bins=50, edgecolor='black', alpha=0.7, color='red')
ax2.axvline(x=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax2.set_xlabel('Error')
ax2.set_ylabel('Frequency')
ax2.set_title('Error Distribution')
ax2.grid(True, alpha=0.3)

# Добавляем статистику на гистограмму
stats_text = f'Mean: {np.mean(errors):.4f}\nStd: {np.std(errors):.4f}\nMax: {np.max(np.abs(errors)):.4f}'
ax2.text(0.02, 0.98, stats_text, transform=ax2.transAxes, 
         verticalalignment='top', 
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

<Figure size 1300x800 with 2 Axes>

In [44]:
from abc import abstractmethod
import casadi as ca
import numpy as np


class Interpolator:
    """Абстрактный интерполятор: y = f(x, params)"""
    @abstractmethod
    def get_np(self):
        """Количество параметров"""
        pass
    
    @abstractmethod
    def evaluate(self, x, params):
        """Вычисляет y по x и параметрам"""
        pass


class LinearInterpolator(Interpolator):
    """y = a * x + b"""
    def __init__(self):
        self.np = 2
    
    def get_np(self):
        return self.np
    
    def evaluate(self, x, params):
        a, b = params[0], params[1]
        return a * x + b


class PiecewiseLinearInterpolator(Interpolator):
    """Кусочно-линейная: на каждом участке свои a и b"""
    def __init__(self, breakpoints):
        self.breakpoints = np.array(breakpoints)
        self.n_segments = len(breakpoints) - 1
        self.np = 2 * self.n_segments
    
    def get_np(self):
        return self.np
    
    def evaluate(self, x, params):
        n_segments = self.n_segments
        breakpoints = self.breakpoints
        
        a = params[0]
        b = params[n_segments]
        
        for i in range(n_segments):
            if i == 0:
                in_segment = x <= breakpoints[i+1]
            elif i == n_segments - 1:
                in_segment = x > breakpoints[i]
            else:
                in_segment = ca.logic_and(x > breakpoints[i], x <= breakpoints[i+1])
            
            a = ca.if_else(in_segment, params[i], a)
            b = ca.if_else(in_segment, params[n_segments + i], b)
        
        return a * x + b


class SmoothPiecewiseLinearInterpolator(Interpolator):
    """Кусочно-линейная с гладкими переходами"""
    def __init__(self, breakpoints, smoothing=10.0):
        self.breakpoints = np.array(breakpoints)
        self.n_segments = len(breakpoints) - 1
        self.np = 2 * self.n_segments
        self.smoothing = smoothing
    
    def get_np(self):
        return self.np
    
    def _sigma(self, x):
        return 1.0 / (1.0 + ca.exp(-self.smoothing * x))
    
    def evaluate(self, x, params):
        n = self.n_segments
        bp = self.breakpoints
        
        a = 0
        b = 0
        
        for i in range(n):
            if i == 0:
                w = 1.0 - self._sigma(x - bp[i+1])
            elif i == n - 1:
                w = self._sigma(x - bp[i])
            else:
                w = self._sigma(x - bp[i]) * (1.0 - self._sigma(x - bp[i+1]))
            
            a += w * params[i]
            b += w * params[n + i]
        
        return a * x + b


class PolynomialInterpolator(Interpolator):
    """y = c0 + c1*x + c2*x^2 + ..."""
    def __init__(self, degree):
        self.degree = degree
        self.np = degree + 1
    
    def get_np(self):
        return self.np
    
    def evaluate(self, x, params):
        result = 0
        for i in range(self.degree + 1):
            result += params[i] * x**i
        return result

In [45]:
class KinematicBycicle(ODESystem):
    def __init__(self, wheelbase, interpolator: Interpolator):
        self.wheelbase = wheelbase
        self.interpolator = interpolator
        super().__init__(nx=1, nu=2, n_theta=interpolator.get_np())

    def observation(self, state, params, input_signals):
        v = input_signals[0]
        steering = input_signals[1]
        
        rwa = self.interpolator.evaluate(steering, params)
        w = v * ca.tan(rwa) / self.wheelbase
        return vertcat(w)
    
    def get_input_signals(self, t):
        return [f_vx(t), f_steer(t)] 
    
interpolator = SmoothPiecewiseLinearInterpolator(breakpoints=np.linspace(f_steer(t).min(), f_steer(t).max(), 20))
system = KinematicBycicle(wheelbase, interpolator)

In [46]:
system.get_input_signals(0), system.n_theta

([array(2.07666667), array(8.82613003)], 38)

In [47]:
regressor = Regressor(system)
theta = np.zeros(system.n_theta)
for i in range(5):
    J, R = regressor.solve(t, state_measured, theta)
    H = J.T@J
    print(theta)
    delta_theta_estimate = np.linalg.pinv(H + 0.01*np.diag(np.diag(H)))@J.T@R
    theta = theta + delta_theta_estimate

[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[ 0.04544514  0.04062799  0.03937362  0.03817583  0.03795959  0.03720348
  0.03783766  0.04221187  0.03654621 -0.18001303  0.03756081  0.04013219
  0.03831185  0.03731732  0.03771164  0.03823278  0.03957789  0.04015252
  0.04591787 -0.32388708 -0.2851644  -0.2446083  -0.20170419 -0.15978957
 -0.12300054 -0.08633757 -0.04570353 -0.03181291 -0.02134624  0.03129018
  0.05244687  0.08811945  0.12545745  0.16300432  0.2041927   0.24131371
  0.29120453  0.32711623]
[ 0.04211203  0.03773011  0.03731324  0.03709698  0.03811499  0.03796915
  0.04026543  0.04730408  0.039534   -0.22269935  0.04112771  0.04458477
  0.04099006  0.03822088  0.03754128  0.03714992  0.03768769  0.03668545
  0.04252893 -0.2697302  -0.25496581 -0.22395533 -0.18811835 -0.14881767
 -0.11569646 -0.07798752 -0.03655387 -0.02944426 -0.02408197  0.02808328
  0.04388121  0.07880846  0.11712767  0.15299855  0.18

In [48]:
fig = plt.figure(figsize=(13, 8))

# Основной график
ax1 = plt.subplot(2, 1, 1)  # 2 строки, 1 столбец, 1-й график
ax1.plot(t, regressor.h_x(t, state_measured, theta), label="regression", alpha=0.8)
ax1.plot(t, state_measured, label="measured", alpha=0.8)
ax1.set_xlabel('Time')
ax1.set_ylabel('Yaw rate')
ax1.set_title('Simple Kinematic model')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Гистограмма ошибок
ax2 = plt.subplot(2, 1, 2)  # 2 строки, 1 столбец, 2-й график

# Вычисляем ошибки
errors = state_measured.flatten() - regressor.h_x(t, state_measured, theta).flatten()

ax2.hist(errors, bins=50, edgecolor='black', alpha=0.7, color='red')
ax2.axvline(x=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax2.set_xlabel('Error')
ax2.set_ylabel('Frequency')
ax2.set_title('Error Distribution')
ax2.grid(True, alpha=0.3)

# Добавляем статистику на гистограмму
stats_text = f'Mean: {np.mean(errors):.4f}\nStd: {np.std(errors):.4f}\nMax: {np.max(np.abs(errors)):.4f}'
ax2.text(0.02, 0.98, stats_text, transform=ax2.transAxes, 
         verticalalignment='top', 
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

<Figure size 1300x800 with 2 Axes>

In [39]:
 system.interpolator.breakpoints

array([-502.6       , -476.18947368, -449.77894737, -423.36842105,
       -396.95789474, -370.54736842, -344.13684211, -317.72631579,
       -291.31578947, -264.90526316, -238.49473684, -212.08421053,
       -185.67368421, -159.26315789, -132.85263158, -106.44210526,
        -80.03157895,  -53.62105263,  -27.21052632,   -0.8       ])

In [85]:
plt.plot( system.interpolator.evaluate( system.interpolator.breakpoints, theta), system.interpolator.breakpoints)

<Figure size 640x480 with 1 Axes>

In [54]:
class KinematicBycicle(ODESystem):
    def __init__(self, wheelbase):
        self.wheelbase = wheelbase
        super().__init__(nx=1, nu=2, n_theta=3)

    def observation(self, state, params, input_signals):
        v = input_signals[0]
        steering = input_signals[1]
        Gr = params[0]
        Gr_sq = params[1]
        offset = params[2]
        rwa = (steering + offset)/(Gr - Gr_sq*(steering + offset)**2) 
        w = v * rwa / self.wheelbase
        return vertcat(w)
    
    def get_input_signals(self, t):
        return [f_vx(t), f_steer(t)] 
    
system = KinematicBycicle(wheelbase)

In [55]:
regressor = Regressor(system)
theta = np.zeros(system.n_theta)
theta[0] = 1.0
for i in range(15):
    J, R = regressor.solve(t, state_measured, theta)
    H = J.T@J
    print(theta)
    delta_theta_estimate = np.linalg.pinv(H + 0.01*np.diag(np.diag(H)))@J.T@R
    theta = theta + delta_theta_estimate

[1. 0. 0.]
[ 1.89680985e+00 -4.42722294e-04  5.61949953e-04]
[ 3.48844651e+00 -1.08983738e-03  7.24592907e-04]
[ 6.05332271e+00 -6.48066067e-04  5.00602833e-04]
[ 9.48200545e+00  5.06888604e-03 -1.21482878e-04]
[ 1.26982981e+01  1.98889319e-02 -1.19779728e-03]
[ 1.42968028e+01  3.46284851e-02 -2.42878368e-03]
[ 1.46045080e+01  3.92248325e-02 -2.96917591e-03]
[ 1.46338035e+01  3.97810288e-02 -3.01999039e-03]
[ 1.46364462e+01  3.98333026e-02 -3.02110929e-03]
[ 1.46366871e+01  3.98380923e-02 -3.02111408e-03]
[ 1.46367091e+01  3.98385300e-02 -3.02111289e-03]
[ 1.46367111e+01  3.98385700e-02 -3.02111276e-03]
[ 1.46367113e+01  3.98385736e-02 -3.02111275e-03]
[ 1.46367113e+01  3.98385740e-02 -3.02111275e-03]


In [56]:
fig = plt.figure(figsize=(13, 8))

# Основной график
ax1 = plt.subplot(2, 1, 1)  # 2 строки, 1 столбец, 1-й график
ax1.plot(t, regressor.h_x(t, state_measured, theta), label="regression", alpha=0.8)
ax1.plot(t, state_measured, label="measured", alpha=0.8)
ax1.set_xlabel('Time')
ax1.set_ylabel('Yaw rate')
ax1.set_title('Simple Kinematic model')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Гистограмма ошибок
ax2 = plt.subplot(2, 1, 2)  # 2 строки, 1 столбец, 2-й график

# Вычисляем ошибки
errors = state_measured.flatten() - regressor.h_x(t, state_measured, theta).flatten()

ax2.hist(errors, bins=50, edgecolor='black', alpha=0.7, color='red')
ax2.axvline(x=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax2.set_xlabel('Error')
ax2.set_ylabel('Frequency')
ax2.set_title('Error Distribution')
ax2.grid(True, alpha=0.3)

# Добавляем статистику на гистограмму
stats_text = f'Mean: {np.mean(errors):.4f}\nStd: {np.std(errors):.4f}\nMax: {np.max(np.abs(errors)):.4f}'
ax2.text(0.02, 0.98, stats_text, transform=ax2.transAxes, 
         verticalalignment='top', 
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

<Figure size 1300x800 with 2 Axes>

In [ ]:
fig = plt.figure(figsize=(13, 8))

# Основной график
ax1 = plt.subplot(2, 1, 1)  # 2 строки, 1 столбец, 1-й график
ax1.plot(t, regressor.h_x(t, state_measured, theta), label="regression", alpha=0.8)
ax1.plot(t, state_measured, label="measured", alpha=0.8)
ax1.set_xlabel('Time')
ax1.set_ylabel('Yaw rate')
ax1.set_title('Simple Kinematic model')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Гистограмма ошибок
ax2 = plt.subplot(2, 1, 2)  # 2 строки, 1 столбец, 2-й график

# Вычисляем ошибки
errors = state_measured.flatten() - regressor.h_x(t, state_measured, theta).flatten()

ax2.hist(errors, bins=50, edgecolor='black', alpha=0.7, color='red')
ax2.axvline(x=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax2.set_xlabel('Error')
ax2.set_ylabel('Frequency')
ax2.set_title('Error Distribution')
ax2.grid(True, alpha=0.3)

# Добавляем статистику на гистограмму
stats_text = f'Mean: {np.mean(errors):.4f}\nStd: {np.std(errors):.4f}\nMax: {np.max(np.abs(errors)):.4f}'
ax2.text(0.02, 0.98, stats_text, transform=ax2.transAxes, 
         verticalalignment='top', 
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.show()

<Figure size 1300x800 with 2 Axes>